# Notebook 03: Bài toán Học máy — Huấn luyện trên 2025 (Q1–Q4) & Dự đoán/Kiểm thử trên 2026 (Q1) (Q6) + Tổng kết Đồ án
**Đề tài:** Phân tích Sự cố Bất lợi Thuốc Thú y (OpenFDA Animal & Veterinary Adverse Events) giai đoạn 2025 và Dự báo cho năm 2026 (Q1)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Nhóm:** Global Ban  
**Chủ sở hữu câu hỏi ML (Q6):** Đinh Công Khang (24120332)

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import (
    TimeSeriesSplit, GridSearchCV, RandomizedSearchCV, cross_val_score, cross_val_predict,
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.inspection import permutation_importance

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import log_row_count_change
from src.visualization import set_project_plot_style
set_project_plot_style()

---
## 1. Định nghĩa Bài toán Dự đoán & Chống Rò rỉ Dữ liệu (ML Requirement 1)

### 1.1. Câu hỏi Q6 & Người dùng mục tiêu (Target User & Decision)
- **Câu hỏi Q6:** Khi một báo cáo sự cố bất lợi mới được gửi đến hệ thống trong **năm 2026 (Q1)**, mô hình học máy được huấn luyện trên dữ liệu lịch sử **2025 (Q1–Q4)** có thể dự đoán chính xác khả năng sự cố đó thuộc nhóm **Nghiêm trọng (`serious_ae = True`)** dựa trên thông tin ban đầu của vật nuôi, thuốc và người báo cáo ở mức độ nào? Ngoài ra, tổng lượng báo cáo nghiêm trọng theo tuần/tháng trong Q1/2026 có khớp với xu hướng dự báo từ năm 2025 hay không?
- **Người dùng mục tiêu (User):** Chuyên viên thẩm định an toàn thuốc tại FDA CVM hoặc hệ thống cảnh báo sớm tại bệnh viện thú y — cần tự động phân loại ưu tiên (triage) các báo cáo có nguy cơ nghiêm trọng cao ngay khi tiếp nhận hồ sơ.

### 1.2. Đặc trưng đầu vào hợp lệ vs. Các cột phải loại bỏ để tránh Rò rỉ Dữ liệu (Data Leakage Prevention)
- **Đặc trưng hợp lệ tại thời điểm dự đoán (Available at prediction time):**
  - Đặc điểm động vật: `species`, `gender`, `reproductive_status`, `age_years`, `weight_kg`, `number_of_animals_affected`.
  - Đặc điểm thuốc & phơi nhiễm: `num_drugs`, `route`, `dosage_form`, `atc_vet_code`, `used_according_to_label`, nhóm hoạt chất chính (`top_active_ingredients`).
  - Thông tin ngữ cảnh tiếp nhận: `primary_reporter`, `type_of_information`, `receive_month` / `quarter`, `reporting_delay_days`.
- **Các cột BẮT BUỘC LOẠI BỎ để tránh Data Leakage:**
  1. `outcome` (`medical_status` như `Died`, `Euthanized`, `Recovered`): Kết quả lâm sàng (Clinical Outcome) xảy ra **sau** sự cố và là thành phần trực tiếp định nghĩa một ca có phải là `serious_ae` hay không.
  2. `treated_for_ae`: Việc có phải can thiệp điều trị cho phản ứng phụ hay không diễn ra sau khi phản ứng đã bộc lộ mức độ nghiêm trọng.
  3. `duration` (thời gian kéo dài của phản ứng): Chỉ biết sau khi sự kiện kết thúc.
  4. Mã định danh (`unique_aer_id_number`, `report_id`) và các nhãn hậu kiểm nội bộ.

---
## 2. Thiết kế Chia Tập dữ liệu theo Thời gian (Time-based Split — ML Requirement 2)

Vì dữ liệu có tính thứ tự thời gian rõ rệt từ đầu năm 2025 đến nửa đầu năm 2026, việc chia ngẫu nhiên (`random train_test_split`) sẽ làm rò rỉ thông tin tương lai (data leakage across time). Do đó, nhóm áp dụng **Time-based Split** nghiêm ngặt:
- **Tập Huấn luyện (Training & Temporal Validation Set):** Toàn bộ báo cáo từ **2025 Q1 đến 2025 Q4** (4 quý).
  - Trong nội bộ 2025 Q1–Q4, sử dụng `TimeSeriesSplit` trên 4 quý (tối đa 3 fold mở rộng: Q1→Q2; Q1–Q2→Q3; Q1–Q3→Q4) để tinh chỉnh siêu tham số (hyperparameter tuning). Kết quả CV cần được diễn giải thận trọng do số điểm thời gian ít.
- **Tập Kiểm thử Thực tế (Out-of-Time Test Set):** Toàn bộ báo cáo của **năm 2026 (Q1)**.
  - Tập 2026 Q1 được giữ kín hoàn toàn trong suốt quá trình thiết kế đặc trưng và tinh chỉnh mô hình, chỉ dùng để **đánh giá đúng 1 lần cuối cùng**.

In [6]:
# ===== Bước 2 (ML Requirement 2): Nạp dữ liệu đã tiền xử lý & tách theo thời gian =====
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REQUIRED_FILES = ["reports.csv", "drugs.csv", "reactions.csv", "outcomes.csv"]
missing = [f for f in REQUIRED_FILES if not (PROCESSED_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Thiếu {missing} trong {PROCESSED_DIR}. Hãy hoàn tất tiền xử lý ở NB01 (Task T14) trước khi chạy phần analysis."
    )

reports = pd.read_csv(PROCESSED_DIR / "reports.csv", dtype=str, keep_default_na=True)
drugs = pd.read_csv(PROCESSED_DIR / "drugs.csv", dtype=str, keep_default_na=True)
for df0 in (reports, drugs):
    df0.columns = [c.strip() for c in df0.columns]

KEY = "unique_aer_id_number"
TRAIN_QUARTERS = ["2025q1", "2025q2", "2025q3", "2025q4"]
TEST_QUARTER = "2026q1"
MISSING_TOKENS = {"", "nan", "none", "unknown", "<na>", "null"}

# 1. Gộp đặc trưng thuốc về mức báo cáo (tránh nhân bản dòng)
drug_agg = drugs.groupby(KEY, as_index=False).size().rename(columns={"size": "num_drugs"})
for col, out in [
    ("route", "n_routes"),
    ("dosage_form", "n_dosage_forms"),
    ("atc_vet_code", "n_atc_vet_codes"),
    ("active_ingredients", "n_active_ingredients"),
]:
    if col in drugs.columns:
        tmp = drugs[[KEY, col]].copy()
        tmp[col] = tmp[col].astype(str).str.strip()
        tmp.loc[tmp[col].str.lower().isin(MISSING_TOKENS), col] = np.nan
        drug_agg[out] = (
            drug_agg[KEY].map(tmp.dropna(subset=[col]).groupby(KEY)[col].nunique())
            .fillna(0).astype(int)
        )

for col, out in [
    ("route", "route_primary"),
    ("dosage_form", "dosage_form_primary"),
    ("active_ingredients", "top_active_ingredients"),
]:
    if col in drugs.columns:
        tmp = drugs[[KEY, col]].copy()
        tmp[col] = tmp[col].astype(str).str.strip()
        tmp.loc[tmp[col].str.lower().isin(MISSING_TOKENS), col] = np.nan
        src_col = tmp.dropna(subset=[col]).drop_duplicates(KEY).set_index(KEY)[col]
        drug_agg[out] = drug_agg[KEY].map(src_col).fillna("Unknown")

log_row_count_change("ML_drug_agg", len(drugs), len(drug_agg),
                     "Gộp đặc trưng thuốc về 1 dòng / báo cáo")
model_df = reports.merge(drug_agg, on=KEY, how="left")
model_df["num_drugs"] = model_df["num_drugs"].fillna(0).astype(int)
model_df["quarter"] = model_df["quarter"].astype(str).str.strip().str.lower()

# 2. Nhãn & mốc thời gian
if "serious" not in model_df.columns:
    model_df["serious"] = (
        model_df["serious_ae"].astype(str).str.strip().str.lower()
        .map({"true": 1, "false": 0, "1": 1, "0": 0})
    )
for c in ["original_receive_date", "onset_date"]:
    if c in model_df.columns:
        model_df[c + "_dt"] = pd.to_datetime(model_df[c], format="%Y%m%d", errors="coerce")
model_df["receive_dt"] = model_df.get(
    "original_receive_date_dt", pd.Series(pd.NaT, index=model_df.index)
)
if "original_receive_date_dt" in model_df.columns and "onset_date_dt" in model_df.columns:
    model_df["reporting_delay_days"] = (
        model_df["original_receive_date_dt"] - model_df["onset_date_dt"]
    ).dt.days
if "original_receive_date_dt" in model_df.columns:
    model_df["receive_month"] = model_df["original_receive_date_dt"].dt.month

quarter_start = {
    q: f"{q[:4]}-{int(q[-1]) * 3 - 2:02d}-01"
    for q in TRAIN_QUARTERS + [TEST_QUARTER]
}
q_start = pd.to_datetime(model_df["quarter"].map(quarter_start))
model_df["receive_week_in_q"] = (
    ((model_df["receive_dt"] - q_start).dt.days // 7 + 1)
    .astype("Int64")
)

# 3. Chuẩn hóa tuổi/cân nặng nếu file processed chưa có cột dẫn xuất
AGE_UNIT_TO_YEARS = {
    "year": 1.0, "years": 1.0, "month": 1 / 12, "months": 1 / 12,
    "week": 1 / 52.18, "weeks": 1 / 52.18, "day": 1 / 365.25, "days": 1 / 365.25,
}
WEIGHT_UNIT_TO_KG = {
    "kilogram": 1.0, "kilograms": 1.0, "kg": 1.0,
    "pound": 0.45359237, "pounds": 0.45359237, "lb": 0.45359237,
    "gram": 0.001, "grams": 0.001, "g": 0.001,
    "ounce": 0.0283495231, "ounces": 0.0283495231, "oz": 0.0283495231,
}
if "age_years" not in model_df.columns and {"age_min", "age_unit"}.issubset(model_df.columns):
    age_raw = pd.to_numeric(model_df["age_min"], errors="coerce")
    factor = model_df["age_unit"].astype(str).str.strip().str.lower().map(AGE_UNIT_TO_YEARS)
    model_df["age_years"] = age_raw * factor
if "weight_kg" not in model_df.columns and {"weight_min", "weight_unit"}.issubset(model_df.columns):
    w_raw = pd.to_numeric(model_df["weight_min"], errors="coerce")
    factor = model_df["weight_unit"].astype(str).str.strip().str.lower().map(WEIGHT_UNIT_TO_KG)
    model_df["weight_kg"] = w_raw * factor

# 4. Danh sách feature hợp lệ tại thời điểm tiếp nhận + danh sách loại bỏ (leakage)
NUMERIC_FEATURES = [c for c in [
    "age_years", "weight_kg", "number_of_animals_affected", "num_drugs",
    "n_routes", "n_dosage_forms", "n_atc_vet_codes", "n_active_ingredients",
    "reporting_delay_days", "receive_month",
] if c in model_df.columns]
CATEGORICAL_FEATURES = [c for c in [
    "species", "gender", "reproductive_status", "primary_reporter",
    "type_of_information", "route_primary", "dosage_form_primary",
    "top_active_ingredients",
] if c in model_df.columns]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
LEAKAGE_COLUMNS = [c for c in [
    "serious_ae", "outcome", "medical_status", "treated_for_ae",
    "duration_value", "duration_unit", KEY, "report_id", "lot_number",
] if c in model_df.columns]
assert not set(FEATURES) & set(LEAKAGE_COLUMNS), "Feature list bị lọt cột leakage!"

# 5. Time-based split: 2025 Q1–Q4 train, 2026 Q1 test (giữ kín đến cuối)
valid = model_df["serious"].notna() & model_df["quarter"].isin(TRAIN_QUARTERS + [TEST_QUARTER])
train_mask = valid & model_df["quarter"].isin(TRAIN_QUARTERS)
test_mask = valid & model_df["quarter"].eq(TEST_QUARTER)
train_order = model_df.loc[train_mask, "receive_dt"].sort_values().index
test_order = model_df.loc[test_mask, "receive_dt"].sort_values().index

X_all = model_df[FEATURES].copy()
X_train = X_all.loc[train_order]
y_train = model_df.loc[train_order, "serious"].astype(int)
X_test_2026 = X_all.loc[test_order]
y_test_2026 = model_df.loc[test_order, "serious"].astype(int)

report_cols = [KEY, "quarter", "receive_month", "receive_week_in_q", "serious"] + [
    c for c in ["species", "primary_reporter", "age_years", "weight_kg"] if c in model_df.columns
]
train_report = model_df.loc[train_order, report_cols].copy()
test_report = model_df.loc[test_order, report_cols].copy()

log_row_count_change("ML_train_2025", int(valid.sum()), len(X_train),
                     "Tách tập train 2025 Q1–Q4 (có nhãn serious_ae)")
log_row_count_change("ML_test_2026Q1", int(valid.sum()), len(X_test_2026),
                     "Tách tập test out-of-time 2026 Q1 (có nhãn serious_ae)")
assert set(model_df.loc[train_order, "quarter"]) == set(TRAIN_QUARTERS)
assert set(model_df.loc[test_order, "quarter"]) == {TEST_QUARTER}
print(f"Train: {X_train.shape[0]:,} dòng × {X_train.shape[1]} feature | Test: {X_test_2026.shape[0]:,} dòng")
display(pd.DataFrame({
    "quarter": model_df.loc[valid, "quarter"].value_counts().reindex(
        TRAIN_QUARTERS + [TEST_QUARTER]).index,
    "n": model_df.loc[valid, "quarter"].value_counts().reindex(
        TRAIN_QUARTERS + [TEST_QUARTER]).values,
}))

---
## 3. Xây dựng Pipeline, Baseline, Huấn luyện & Tinh chỉnh Siêu tham số (ML Requirements 3 & 4)

### 3.1. Kiến trúc `scikit-learn Pipeline`
- Toàn bộ các bước xử lý khuyết (`SimpleImputer`), chuẩn hóa thang đo (`StandardScaler`) và mã hóa biến phân loại (`OneHotEncoder(handle_unknown='ignore')`) đều nằm bên trong `ColumnTransformer` và `Pipeline` để tránh rò rỉ thống kê từ tập Validation/Test2026 vào tập Train.

### 3.2. Các mô hình so sánh
1. **Trivial Baseline:** `DummyClassifier(strategy='most_frequent')` (và baseline theo tỷ lệ lớp năm 2025).
2. **Mô hình tuyến tính:** `LogisticRegression(max_iter=1000, class_weight='balanced')`.
3. **Mô hình phi tuyến / Ensemble:** `HistGradientBoostingClassifier` và/hoặc `RandomForestClassifier`.

### 3.3. Thước đo đánh giá (Evaluation Metrics Justification)
- Chọn **PR-AUC (Average Precision)** và **ROC-AUC** làm thước đo chính khi tinh chỉnh với `TimeSeriesSplit`, kèm theo **F1-score / Recall** ở ngưỡng tối ưu.
- **Lý do:** Trong bài toán cảnh giác dược, việc bỏ sót một sự cố nghiêm trọng (False Negative) nguy hiểm hơn việc cảnh báo nhầm (False Positive), đồng thời tỷ lệ lớp có thể biến động giữa các năm nên `Accuracy` đơn thuần không phản ánh đúng chất lượng phân tách xác suất.

In [7]:
# ===== Bước 3–4 (ML Requirements 3 & 4): Pipeline, baseline, CV & đánh giá cuối =====
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATURES),
    ("cat", categorical_pipe, CATEGORICAL_FEATURES),
], remainder="drop")

models = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "logistic": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "hist_gb": HistGradientBoostingClassifier(random_state=42),
    "random_forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced_subsample",
        random_state=42, n_jobs=-1,
    ),
}
pipelines = {
    name: Pipeline([("prep", preprocessor), ("clf", clf)])
    for name, clf in models.items()
}

tscv = TimeSeriesSplit(n_splits=3)
cv_rows = []
for name, pipe in pipelines.items():
    scores = cross_val_score(
        pipe, X_train, y_train, cv=tscv, scoring="average_precision", n_jobs=-1
    )
    cv_rows.append({
        "model": name,
        "cv_pr_auc_mean": scores.mean(),
        "cv_pr_auc_std": scores.std(),
    })
cv_summary = pd.DataFrame(cv_rows).sort_values("cv_pr_auc_mean", ascending=False)
display(cv_summary)

best_name = cv_summary.iloc[0]["model"]
param_grids = {
    "logistic": {"clf__C": [0.1, 1.0, 10.0]},
    "hist_gb": {
        "clf__learning_rate": [0.05, 0.1],
        "clf__max_leaf_nodes": [15, 31],
    },
    "random_forest": {
        "clf__n_estimators": [200],
        "clf__max_depth": [None, 12],
    },
}
search = GridSearchCV(
    pipelines[best_name], param_grids.get(best_name, {}),
    scoring="average_precision", cv=tscv, n_jobs=-1, refit=True,
)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print(f"Mô hình được chọn: {best_name} | tham số tốt nhất: {search.best_params_}")

# Chọn ngưỡng triage từ out-of-fold trên 2025; KHÔNG tinh chỉnh trên tập test 2026
oof_score = cross_val_predict(
    best_model, X_train, y_train, cv=tscv, method="predict_proba", n_jobs=-1
)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_train, oof_score)
f1_oof = (
    2 * precision[:-1] * recall[:-1]
    / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
)
best_threshold = float(thresholds[int(np.nanargmax(f1_oof))])

# Đánh giá đúng 1 lần trên 2026 Q1
y_score_2026 = best_model.predict_proba(X_test_2026)[:, 1]
y_pred_2026 = (y_score_2026 >= best_threshold).astype(int)
print(f"Ngưỡng triage (từ CV 2025): {best_threshold:.3f}")

---
## 4. Phân tích Lỗi trên Tập Kiểm thử 2026 Q1 (Error Analysis — ML Requirement 5)
- Đánh giá độ chính xác của mô hình trên quý đầu của năm 2026 (`2026 Q1`) để kiểm tra hiện tượng **Temporal Drift / Concept Drift**.
- Phân tích tỷ lệ lỗi (False Positives & False Negatives) theo từng phân nhóm: Loài (`Dog`, `Cat`, `Cattle`, `Horse`), Đối tượng báo cáo (`Veterinarian` vs. `Animal Owner`), và các khoảng giá trị tuổi/cân nặng.
- Trích xuất một số dòng báo cáo thực tế năm 2026 mà mô hình dự đoán sai nặng nhất để giải thích nguyên nhân.
- Đối chiếu **tổng lượng báo cáo nghiêm trọng theo tuần/tháng** của 2026 Q1 (thực tế và dự đoán) với cùng kỳ 2025 để trả lời phần cuối của câu hỏi Q6.

In [8]:
# ===== Bước 4 (ML Requirement 5): Đánh giá & phân tích lỗi trên 2026 Q1 =====
metrics = {
    "PR_AUC (average precision)": average_precision_score(y_test_2026, y_score_2026),
    "ROC_AUC": roc_auc_score(y_test_2026, y_score_2026),
    "Precision": precision_score(y_test_2026, y_pred_2026, zero_division=0),
    "Recall": recall_score(y_test_2026, y_pred_2026, zero_division=0),
    "F1": f1_score(y_test_2026, y_pred_2026, zero_division=0),
}
display(pd.DataFrame([metrics]).T.rename(columns={0: "giá_trị"}).round(4))

cm = confusion_matrix(y_test_2026, y_pred_2026)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=axes[0])
axes[0].set_xlabel("Dự đoán"); axes[0].set_ylabel("Thực tế")
axes[0].set_title("Confusion matrix (2026 Q1)", fontweight="bold")
RocCurveDisplay.from_predictions(y_test_2026, y_score_2026, ax=axes[1])
axes[1].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test_2026, y_score_2026, ax=axes[2])
axes[2].set_title("Precision–Recall curve")
plt.tight_layout()
plt.show()

error_df = test_report.copy()
error_df["y_score"] = y_score_2026
error_df["y_pred"] = y_pred_2026
error_df["is_error"] = error_df["serious"].ne(error_df["y_pred"])

def error_by_group(frame, col):
    g = frame.groupby(col, observed=True)
    fn = ((frame["serious"].eq(1) & frame["y_pred"].eq(0))
          .groupby(frame[col]).sum())
    fp = ((frame["serious"].eq(0) & frame["y_pred"].eq(1))
          .groupby(frame[col]).sum())
    out = pd.DataFrame({
        "n": g.size(),
        "n_error": g["is_error"].sum(),
        "FN": fn,
        "FP": fp,
    }).fillna(0).astype(int)
    out["error_rate_pct"] = out["n_error"] / out["n"] * 100
    return out.sort_values("error_rate_pct", ascending=False)

for col in ["species", "primary_reporter"]:
    if col in error_df.columns:
        print(f"\nPhân rã lỗi theo {col}:")
        display(error_by_group(error_df.assign(**{col: error_df[col].fillna("Unknown")}), col).round(2))

for col in ["age_years", "weight_kg"]:
    if col in error_df.columns:
        try:
            edges = train_report[col].quantile([0, 0.25, 0.5, 0.75, 1]).drop_duplicates().values
            if len(edges) >= 3:
                binned = error_df.assign(
                    _bin=pd.cut(error_df[col], bins=edges, include_lowest=True, duplicates="drop").astype(str)
                )
                print(f"\nPhân rã lỗi theo {col} (quartile của tập train):")
                display(error_by_group(binned, "_bin").round(2))
        except (ValueError, TypeError):
            print(f"\nBỏ qua phân rã lỗi theo {col}: không đủ giá trị để chia khoảng.")

worst_fn = error_df[error_df["serious"].eq(1) & error_df["y_pred"].eq(0)] \
    .nsmallest(5, "y_score")
worst_fp = error_df[error_df["serious"].eq(0) & error_df["y_pred"].eq(1)] \
    .nlargest(5, "y_score")
show_cols = [c for c in [KEY, "quarter", "species", "primary_reporter", "age_years", "weight_kg", "y_score"]
             if c in error_df.columns]
print("\nCa nghiêm trọng bị bỏ sót (FN) có xác suất thấp nhất:")
display(worst_fn[show_cols])
print("\nCa không nghiêm trọng bị cảnh báo nhầm (FP) có xác suất cao nhất:")
display(worst_fp[show_cols])

In [ ]:
# ===== Bước 4b: Tổng lượng báo cáo nghiêm trọng theo tuần/tháng (phần cuối Q6) =====
def weekly_serious_counts(frame, quarter):
    sub = frame[frame["quarter"].eq(quarter)]
    return sub.groupby("receive_week_in_q", observed=True)["serious"] \
        .agg(n_reports="count", n_serious="sum")

w25 = weekly_serious_counts(train_report, "2025q1")
w26 = weekly_serious_counts(test_report, "2026q1")
weekly_cmp = w25.join(w26, lsuffix="_2025q1", rsuffix="_2026q1", how="outer") \
    .fillna(0).astype(int)
print("Số báo cáo và số ca serious theo tuần trong Q1:")
display(weekly_cmp)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(weekly_cmp.index, weekly_cmp["n_serious_2025q1"], marker="o", label="2025 Q1 (cùng kỳ)")
ax.plot(weekly_cmp.index, weekly_cmp["n_serious_2026q1"], marker="o", label="2026 Q1 (thực tế)")
ax.set_xlabel("Tuần trong quý")
ax.set_ylabel("Số báo cáo serious_ae")
ax.set_title("Tổng lượng báo cáo nghiêm trọng theo tuần", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

pred_by_week = (
    test_report.assign(y_pred=y_pred_2026)
    .groupby("receive_week_in_q", observed=True)["y_pred"].sum()
    .rename("n_serious_du_bao")
)
weekly_model = weekly_cmp[["n_serious_2026q1"]].join(pred_by_week, how="outer") \
    .fillna(0).astype(int)
weekly_model["chenh_thuc_te_tru_du_bao"] = (
    weekly_model["n_serious_2026q1"] - weekly_model["n_serious_du_bao"]
)
print("Thực tế 2026 Q1 vs số ca mô hình dự báo (theo tuần):")
display(weekly_model)

q1_frames = pd.concat([
    train_report[train_report["quarter"].eq("2025q1")].assign(nhom="2025 Q1"),
    test_report.assign(nhom="2026 Q1"),
])
monthly_cmp = (
    q1_frames.groupby(["receive_month", "nhom"])["serious"]
    .agg(n_reports="count", n_serious="sum")
    .unstack("nhom")
)
monthly_cmp.columns = [f"{a}_{b}" for a, b in monthly_cmp.columns]
print("Tổng hợp theo tháng của Q1:")
display(monthly_cmp.fillna(0).astype(int))
print("Lưu ý: 2026 Q1 là snapshot (posting delay); kết luận 'khớp/không khớp xu hướng' chỉ được rút ra sau khi đọc các bảng trên cùng bối cảnh thiên lệch báo cáo.")

---
## 5. Giải thích Mô hình & Trả lời Câu hỏi Nghiên cứu Q6 (Model Interpretation — ML Requirement 6)
- Sử dụng **Permutation Importance** trên tập kiểm thử 2026 (Q1) và hệ số mô hình để xác định những đặc trưng nào đóng góp lớn nhất vào dự đoán `serious_ae`.
- Kết luận rõ ràng bằng số liệu thực nghiệm và ý nghĩa thực tiễn.
- Kết luận cần trả lời **cả hai phần** của Q6: độ chính xác dự báo ở mức độ nào và tổng lượng báo cáo nghiêm trọng theo tuần/tháng trong 2026 Q1 có khớp xu hướng 2025 hay không.

In [9]:
# ===== Bước 5 (ML Requirement 6): Giải thích mô hình & trả lời Q6 =====
perm = permutation_importance(
    best_model, X_test_2026, y_test_2026,
    scoring="average_precision", n_repeats=10, random_state=42, n_jobs=-1,
)
imp_df = pd.DataFrame({
    "feature": X_test_2026.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
display(imp_df)

fig, ax = plt.subplots(figsize=(8, 5))
top_imp = imp_df.head(12).sort_values("importance_mean")
ax.barh(top_imp["feature"], top_imp["importance_mean"],
        xerr=top_imp["importance_std"], color="#4c78a8", alpha=0.85)
ax.set_xlabel("Giảm PR-AUC khi xáo trộn (permutation importance)")
ax.set_title("Đặc trưng quan trọng nhất trên tập test 2026 Q1", fontweight="bold")
plt.tight_layout()
plt.show()

if best_name == "logistic":
    names = best_model.named_steps["prep"].get_feature_names_out()
    coefs = best_model.named_steps["clf"].coef_[0]
    coef_df = pd.DataFrame({"feature": names, "coef": coefs})
    display(coef_df.reindex(coef_df["coef"].abs().sort_values(ascending=False).index).head(15))

print("Khung kết luận Q6 (điền số liệu sau khi chạy thực nghiệm):")
print("- Hiệu năng dự báo serious_ae trên 2026 Q1: PR-AUC = ..., ROC-AUC = ..., Recall/F1 tại ngưỡng ... = ...")
print("- Đặc trưng đóng góp lớn nhất: ... (cần đọc cùng confounding và thiên lệch báo cáo).")
print("- Drift train 2025 vs test 2026 Q1: ...")
print("- Đối chiếu tổng lượng báo cáo nghiêm trọng theo tuần/tháng: ... → có/không khớp xu hướng 2025 (kèm giải thích posting delay).")

---
## 6. Tổng kết Đồ án & Phản ánh Cá nhân (Summary and Reflection — Mục 3.5)

### 6.1. Các phát hiện quan trọng nhất (Key Findings — 3 đến 5 insights)
- *(Tổng hợp kết quả nổi bật nhất từ Q1–Q6 sau khi chạy thực nghiệm, nhấn mạnh phát hiện bất ngờ nhất)*

### 6.2. Hạn chế của Dữ liệu, Phương pháp và Phạm vi (Limitations)
- Hạn chế của dữ liệu báo cáo tự nguyện openFDA (thiếu mẫu số tổng đàn, thiên lệch báo cáo, dữ liệu chưa xác thực nhân quả).
- Hạn chế do rút gọn dữ liệu huấn luyện còn 4 quý năm 2025: ít điểm thời gian, CV kém ổn định và không đủ cơ sở khẳng định chu kỳ mùa vụ dài hạn.
- Hạn chế về thời gian của năm 2026 (mới có Q1 nên chưa phản ánh trọn vẹn chu kỳ mùa thu-đông của Q3–Q4 năm 2026).

### 6.3. Hướng phát triển tiếp theo (Future Work)
- Bổ sung dữ liệu Q3 và Q4 năm 2026 khi FDA phát hành để kiểm chứng tính mùa vụ cả năm.
- Kết hợp thêm từ điển phân loại dược lý ATCvet hoặc dữ liệu nhãn thuốc (openFDA Animal & Veterinary Drug Labeling).

### 6.4. Phản ánh Cá nhân (Individual Reflections)
- **Đinh Công Khang (24120332):** *(Khó khăn gặp phải, cách vượt qua, bài học rút ra, cách sử dụng công cụ LLM)*
- **Trần Ngô Uyên Nhi (24120210):** *(Khó khăn gặp phải, cách vượt qua, bài học rút ra, cách sử dụng công cụ LLM)*
- **Huỳnh Tuấn Kiệt (24120356):** *(Khó khăn gặp phải, cách vượt qua, bài học rút ra, cách sử dụng công cụ LLM)*